# 00 - Setup: Carga de Dados no SQL Server

Este notebook carrega os arquivos CSV da pasta `data/` para o banco de dados **SeguroDB** no SQL Server 2025.

**Pré-requisitos:**
- Docker Compose rodando (`docker compose up -d`)
- SQL Server acessível na porta `1433`
- Driver ODBC 18 instalado (`sudo apt install unixodbc-dev`)

## 1. Configuração e Conexão

In [3]:
import os
import pandas as pd
import pyodbc
from dotenv import load_dotenv

load_dotenv(override=True)

DB_SERVER   = os.getenv('DB_SERVER')
DB_PORT     = os.getenv('DB_PORT')
DB_USER     = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_DATABASE = os.getenv('DB_DATABASE')

print(f'Servidor: {DB_SERVER}:{DB_PORT}')
print(f'Database: {DB_DATABASE}')

Servidor: localhost:1433
Database: SeguroDB


In [4]:
# Conexão ao SQL Server (master) para criar o database
conn_master = pyodbc.connect(
    f'DRIVER={{ODBC Driver 18 for SQL Server}};'
    f'SERVER={DB_SERVER},{DB_PORT};'
    f'UID={DB_USER};'
    f'PWD={DB_PASSWORD};'
    f'TrustServerCertificate=yes;',
    autocommit=True
)
cursor_master = conn_master.cursor()
print('Conectado ao SQL Server (master) com sucesso!')

Conectado ao SQL Server (master) com sucesso!


## 2. Criar Database SeguroDB

In [5]:
# Criar o database se não existir
cursor_master.execute(f"""
    IF NOT EXISTS (SELECT name FROM sys.databases WHERE name = '{DB_DATABASE}')
    BEGIN
        CREATE DATABASE [{DB_DATABASE}]
    END
""")
print(f'Database [{DB_DATABASE}] criado/verificado com sucesso!')
cursor_master.close()
conn_master.close()

Database [SeguroDB] criado/verificado com sucesso!


In [6]:
# Conectar ao database SeguroDB
conn = pyodbc.connect(
    f'DRIVER={{ODBC Driver 18 for SQL Server}};'
    f'SERVER={DB_SERVER},{DB_PORT};'
    f'DATABASE={DB_DATABASE};'
    f'UID={DB_USER};'
    f'PWD={DB_PASSWORD};'
    f'TrustServerCertificate=yes;',
    autocommit=True
)
cursor = conn.cursor()
print(f'Conectado ao [{DB_DATABASE}] com sucesso!')

Conectado ao [SeguroDB] com sucesso!


## 3. Criar Tabelas

In [7]:
# DDL - Criação das tabelas
ddl_statements = [
    # Tabelas de domínio (sem FK)
    """
    IF NOT EXISTS (SELECT * FROM sys.tables WHERE name = 'regiao')
    CREATE TABLE regiao (
        cd_regiao   INT PRIMARY KEY,
        nm_regiao   VARCHAR(50) NOT NULL
    )
    """,
    """
    IF NOT EXISTS (SELECT * FROM sys.tables WHERE name = 'estado')
    CREATE TABLE estado (
        cd_estado   INT PRIMARY KEY,
        cd_regiao   INT NOT NULL,
        nm_estado   VARCHAR(100) NOT NULL,
        sigla_uf    CHAR(2) NOT NULL
    )
    """,
    """
    IF NOT EXISTS (SELECT * FROM sys.tables WHERE name = 'municipio')
    CREATE TABLE municipio (
        cd_municipio INT PRIMARY KEY,
        nm_municipio VARCHAR(200) NOT NULL,
        cd_estado    INT NOT NULL
    )
    """,
    """
    IF NOT EXISTS (SELECT * FROM sys.tables WHERE name = 'marca')
    CREATE TABLE marca (
        cd_marca    INT PRIMARY KEY,
        nm_marca    VARCHAR(100) NOT NULL
    )
    """,
    """
    IF NOT EXISTS (SELECT * FROM sys.tables WHERE name = 'modelo')
    CREATE TABLE modelo (
        cd_modelo   INT PRIMARY KEY,
        cd_marca    INT NOT NULL,
        nm_modelo   VARCHAR(100) NOT NULL
    )
    """,
    """
    IF NOT EXISTS (SELECT * FROM sys.tables WHERE name = 'cliente')
    CREATE TABLE cliente (
        cd_cliente      INT PRIMARY KEY,
        nome            VARCHAR(200) NOT NULL,
        cpf             VARCHAR(14) NOT NULL,
        sexo            CHAR(1),
        dt_nascimento   DATE
    )
    """,
    """
    IF NOT EXISTS (SELECT * FROM sys.tables WHERE name = 'endereco')
    CREATE TABLE endereco (
        cd_cliente      INT NOT NULL,
        cd_municipio    INT NOT NULL,
        ds_endereco     VARCHAR(300),
        nr_endereco     INT,
        bairro          VARCHAR(200)
    )
    """,
    """
    IF NOT EXISTS (SELECT * FROM sys.tables WHERE name = 'telefone')
    CREATE TABLE telefone (
        cd_cliente      INT NOT NULL,
        nr_telefone     VARCHAR(20) NOT NULL
    )
    """,
    """
    IF NOT EXISTS (SELECT * FROM sys.tables WHERE name = 'carro')
    CREATE TABLE carro (
        placa       VARCHAR(10) NOT NULL,
        cd_modelo   INT NOT NULL,
        chassi      BIGINT,
        ano         INT,
        cor         VARCHAR(30)
    )
    """,
    """
    IF NOT EXISTS (SELECT * FROM sys.tables WHERE name = 'apolice')
    CREATE TABLE apolice (
        cd_apolice          INT PRIMARY KEY,
        cd_cliente          INT NOT NULL,
        dt_inicio_vigencia  DATETIME,
        dt_fim_vigencia     DATETIME,
        vl_cobertura        DECIMAL(12,2),
        vl_franquia         DECIMAL(12,2),
        placa               VARCHAR(10)
    )
    """,
    """
    IF NOT EXISTS (SELECT * FROM sys.tables WHERE name = 'sinistro')
    CREATE TABLE sinistro (
        cd_sinistro     INT PRIMARY KEY,
        placa           VARCHAR(10),
        dt_sinistro     DATETIME,
        local_sinistro  VARCHAR(200),
        condutor        VARCHAR(200)
    )
    """
]

for ddl in ddl_statements:
    cursor.execute(ddl)
    
print('Todas as tabelas foram criadas com sucesso!')

Todas as tabelas foram criadas com sucesso!


## 4. Carregar Dados dos CSVs

In [8]:
# Ordem de carga (respeitar dependências)
tabelas = [
    'regiao', 'estado', 'municipio', 'marca', 'modelo',
    'cliente', 'endereco', 'telefone', 'carro', 'apolice', 'sinistro'
]

data_dir = os.path.join(os.getcwd(), 'data')

for tabela in tabelas:
    csv_path = os.path.join(data_dir, f'{tabela}.csv')
    
    # Verificar se a tabela já tem dados
    cursor.execute(f'SELECT COUNT(*) FROM {tabela}')
    count = cursor.fetchone()[0]
    
    if count > 0:
        print(f'  ⏭️  {tabela}: já contém {count} registros, pulando...')
        continue
    
    # Ler CSV
    df = pd.read_csv(csv_path)
    
    # Limpar espaços em colunas string
    for col in df.select_dtypes(include=['str', 'object']).columns:
        df[col] = df[col].str.strip()
    
    # Inserir dados via executemany
    cols = ', '.join(df.columns)
    placeholders = ', '.join(['?' for _ in df.columns])
    insert_sql = f'INSERT INTO {tabela} ({cols}) VALUES ({placeholders})'
    
    # Converter NaN para None
    data = [tuple(None if pd.isna(v) else v for v in row) for row in df.itertuples(index=False)]
    
    # Inserir em lotes de 1000
    batch_size = 2000
    for i in range(0, len(data), batch_size):
        batch = data[i:i+batch_size]
        cursor.executemany(insert_sql, batch)
    
    print(f'  ✅ {tabela}: {len(data)} registros inseridos')

print('\n🎉 Carga de dados concluída!')

  ✅ regiao: 5 registros inseridos
  ✅ estado: 27 registros inseridos
  ✅ municipio: 5570 registros inseridos
  ✅ marca: 10 registros inseridos
  ✅ modelo: 100 registros inseridos
  ✅ cliente: 20010 registros inseridos
  ✅ endereco: 20010 registros inseridos
  ✅ telefone: 20010 registros inseridos
  ✅ carro: 10002 registros inseridos
  ✅ apolice: 10000 registros inseridos
  ✅ sinistro: 10000 registros inseridos

🎉 Carga de dados concluída!


## 5. Validação

In [9]:
# Verificar contagem de registros em cada tabela
print(f'{"Tabela":<20} {"Registros":>10}')
print('-' * 32)

for tabela in tabelas:
    cursor.execute(f'SELECT COUNT(*) FROM {tabela}')
    count = cursor.fetchone()[0]
    print(f'{tabela:<20} {count:>10}')

print('\n✅ Validação concluída!')

Tabela                Registros
--------------------------------
regiao                        5
estado                       27
municipio                  5570
marca                        10
modelo                      100
cliente                   20010
endereco                  20010
telefone                  20010
carro                     10002
apolice                   10000
sinistro                  10000

✅ Validação concluída!


In [10]:
# Amostra de dados de algumas tabelas
for tabela in ['regiao', 'marca', 'cliente']:
    print(f'\n--- {tabela.upper()} (primeiros 5 registros) ---')
    df_sample = pd.read_sql(f'SELECT TOP 5 * FROM {tabela}', conn)
    print(df_sample.to_string(index=False))


--- REGIAO (primeiros 5 registros) ---
 cd_regiao    nm_regiao
         1        NORTE
         2     NORDESTE
         3      SUDESTE
         4          SUL
         5 CENTRO-OESTE

--- MARCA (primeiros 5 registros) ---
 cd_marca   nm_marca
        1  CHEVROLET
        2 VOLKSWAGEN
        3       FIAT
        4       FORD
        5     TOYOTA

--- CLIENTE (primeiros 5 registros) ---
 cd_cliente                     nome         cpf sexo dt_nascimento
          1     MARISA MELO OLIVEIRA 54367845687    F    2000-09-06
          2  MURILO CARVALHO CARDOSO 34563456566    M    1989-06-26
          3 VINICIUS ROCHA RODRIGUES 83946466633    M    1990-01-25
          4     CAROLINA ROCHA GOMES  9837746366    F    1996-04-21
          5      ALINE SANTOS CASTRO 99934377292    F    1962-07-04


/tmp/ipykernel_12368/948127430.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_sample = pd.read_sql(f'SELECT TOP 5 * FROM {tabela}', conn)
/tmp/ipykernel_12368/948127430.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_sample = pd.read_sql(f'SELECT TOP 5 * FROM {tabela}', conn)
/tmp/ipykernel_12368/948127430.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_sample = pd.read_sql(f'SELECT TOP 5 * FROM {tabela}', conn)


In [11]:
# Encerrar conexão
cursor.close()
conn.close()
print('Conexão encerrada.')

Conexão encerrada.
